# SpectralGuard Phase 4 — H100 adaptive endgame
Select an H100 runtime and click **Run all** once. This notebook validates the frozen CUDA path, resumes atomically, stops before the 2.4-hour export reserve, and creates either `phase4_experiments.zip` or `phase4_handoff.tar.gz`.

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
if not Path('configs/phase4.yaml').is_file():
    os.chdir('..')
assert Path('configs/phase4.yaml').is_file(), f'Repository root not found from {Path.cwd()}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, 'scripts/install_lightning_mamba_kernels.py', '--cuda-smoke'], check=True)
print('Finished')

In [ ]:
import torch
from src.experiments.phase4_common import phase4_preflight
assert torch.cuda.is_available() and 'H100' in torch.cuda.get_device_name(0).upper()
phase4_preflight(require_h100=True)
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
print('Finished')

In [ ]:
from src.experiments.phase4_runner import run_automatic_phase4
phase4_report = run_automatic_phase4()
assert phase4_report['status'] in {'COMPLETE', 'TIMEBOX_HANDOFF'}
print(phase4_report)
print('Finished')

In [ ]:
from pathlib import Path
complete = phase4_report['status'] == 'COMPLETE'
result = Path('Experiments/exports/phase4_experiments.zip' if complete else 'Experiments/exports/phase4_handoff.tar.gz').resolve()
assert result.is_file() and result.stat().st_size > 0
digest = hashlib.sha256(result.read_bytes()).hexdigest()
print(f'DOWNLOAD READY: {result}')
print(f'SHA256: {digest}')
print('STOP THE H100 NOW. Run Notebook 04 on CPU.' if complete else 'STOP THE H100 NOW. Upload this handoff to the next session.')
print('Finished')